# Trening NER na Colabie (T4) — porównanie 3 modeli

Fine-tuning na datasecie medycznym z encjami PII: `output/ner_dataset.jsonl`
(8770 próbek, **9 typów encji**, angielskie etykiety: PERSON, DISEASE, DRUG,
TEST, HOSPITAL + ADDRESS, DATE, PESEL, PHONE → 19 klas IOB2).

| Model | Rozmiar | Oś porównania |
|---|---|---|
| `allegro/herbert-base-cased` | 124M | baseline PL (BERT) |
| `sdadas/polish-roberta-base-v2` | 124M | BERT vs RoBERTa |
| `xlm-roberta-base` | 278M | polski vs multilingual |

**Przed startem:** Runtime → Change runtime type → **T4 GPU**.

Każdy model ewaluowany dwukrotnie: na odłożonym **test splicie** (ta sama fabryka
co train — zawyża) oraz na **golden secie** z `test/` (niezależny, LLM-generated —
realna miara generalizacji). Wyniki w W&B (projekt `nlp-ner`).

In [ ]:
!nvidia-smi

In [ ]:
# idempotentne: re-run komórki nie zagnieżdża klonów (ścieżki absolutne)
import os
if not os.path.exists("/content/nlp-ner"):
    !git clone https://github.com/marek-olejniczak/nlp-ner.git /content/nlp-ner
%cd /content/nlp-ner
!git pull

In [ ]:
# torch jest preinstalowany na Colabie
!pip install -q transformers datasets seqeval accelerate wandb tqdm

In [ ]:
import wandb
wandb.login()  # wklej API key z https://wandb.ai/authorize

In [ ]:
# (model, batch, grad_accum) — wszystkie base'y mieszczą batch 16 na 16GB T4
MODELS = [
    ("allegro/herbert-base-cased", 16, 1),
    ("sdadas/polish-roberta-base-v2", 16, 1),
    ("xlm-roberta-base", 16, 1),
]

## Trening + ewaluacja w pętli

Każdy model: fine-tuning (3 epoki, lr 2e-5, fp16, najlepszy checkpoint wg F1
na walidacji) → ewaluacja na test splicie i na golden secie.
Paski tqdm pokazują postęp na bieżąco.

Najlepszy checkpoint każdego modelu jest automatycznie logowany jako **artefakt W&B**
(`<model>-ner`, type=model) — sesja Colaba jest ulotna, więc model nie zginie nawet
jak runtime padnie. Podgląd: zakładka *Artifacts* w projekcie `nlp-ner` na wandb.ai.

In [ ]:
DATA = "output/ner_dataset.jsonl"
GOLDEN = "test/dataset_1.json"  # niezależny golden set (inline markup)

for model, bs, accum in MODELS:
    short = model.split("/")[-1]
    print(f"\n{'='*70}\n  {model}\n{'='*70}")
    !python -m training.train --model {model} --data {DATA} --batch-size {bs} --grad-accum {accum} --fp16
    print("--- test split (ta sama fabryka co train) ---")
    !python -m training.evaluate --checkpoint models/{short}/best --data {DATA}
    print("--- golden set (niezależny) ---")
    !python -m training.eval_set --input {GOLDEN} --checkpoint models/{short}/best

## Tabela zbiorcza

Micro F1 = wszystkie encje do jednego worka (dominują częste klasy).
Macro F1 = średnia po typach encji (wrażliwa na słabe klasy, np. DRUG).
Kolumna **golden** to F1 na niezależnym secie — spodziewaj się, że będzie niższy
od test split (split pochodzi z tej samej fabryki szablonów co train).

In [ ]:
import json
from pathlib import Path

def micro_f1(path):
    if not Path(path).exists():
        return None
    return json.loads(Path(path).read_text())["micro avg"]["f1-score"]

print(f"{'model':32s} {'split F1':>9s} {'golden F1':>10s}")
print("-" * 53)
for model, _, _ in MODELS:
    short = model.split("/")[-1]
    split = micro_f1(f"models/{short}/best/eval_report_test.json")
    golden = micro_f1(f"models/{short}/best/eval_report_golden.json")
    fmt = lambda x: f"{x:.4f}" if x is not None else "   —  "
    print(f"{short:32s} {fmt(split):>9s} {fmt(golden):>10s}")

## Pobranie wytrenowanych modeli

Sesja Colaba jest ulotna — pobierz checkpointy (i raporty JSON) zanim znikną.
(~500MB na model base; alternatywnie Google Drive albo `huggingface_hub`.)

In [ ]:
# pobierz zwycięski model — podmień nazwę wg tabeli wyżej
BEST = "herbert-base-cased"
!zip -rq {BEST}-ner.zip models/{BEST}/best
from google.colab import files
files.download(f"{BEST}-ner.zip")